as we want to capture all the changes for every account we wont be checking for duplicates for this dimension

In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
accounts_path = "/Volumes/coreview/landing/organizations_data/data/crm_accounts.csv"

In [0]:
accounts_df = spark.read.format('csv').option('header', 'true').load(accounts_path)

Adding timestamp columns

In [0]:
accounts_df = (accounts_df.withColumn("Ingestion_timestamp", F.date_format(F.current_timestamp(), "yyyy-MM-dd HH:mm:ss"))
                        .withColumn("Ingestion_date", F.to_date("Ingestion_timestamp"))
)

In [0]:
# display(accounts_df)

In [0]:
acc_table_name = "coreview.bronze.crm_accounts"

if spark.catalog.tableExists(acc_table_name):
    accounts_df.write.format("delta").mode("append").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(acc_table_name)
else:
    accounts_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(acc_table_name)


In [0]:
# %%sql
# select * from coreview.bronze.crm_accounts

as we want to capture all the changes for every organisation we wont be checking for duplicates for this dimension

In [0]:
organisations_path = '/Volumes/coreview/landing/organizations_data/data/organizations/'

In [0]:
# JSON-lines files: one object per line, so no multiLine option needed.
# Schema is inferred automatically - no schema definition required.
org_df = spark.read.json(organisations_path)

In [0]:
display(org_df)

used Genie to build the below method

In [0]:
from pyspark.sql.types import StructType

def flatten_df(df):
    """Flatten nested structs into parent_child columns (arrays are left intact)."""
    while any(isinstance(df.schema[c].dataType, StructType) for c in df.columns):
        cols = []
        for c in df.columns:
            if isinstance(df.schema[c].dataType, StructType):
                for f in df.schema[c].dataType.fields:
                    cols.append(F.col(f"{c}.{f.name}").alias(f"{c}_{f.name}"))
            else:
                cols.append(F.col(c))
        df = df.select(*cols)
    return df

org_flat = flatten_df(org_df)
org_flat.printSchema()

In [0]:
display(org_flat)

In [0]:
# One row per addon. explode_outer keeps orgs that have no addons (license_addon = null).
org_exploded = (org_flat
    .withColumn("license_addon", F.explode_outer(F.col("license_addons")))
    .drop("license_addons")
)

In [0]:
display(org_exploded)

In [0]:
org_exploded = (org_exploded.withColumn("Ingestion_timestamp", F.date_format(F.current_timestamp(), "yyyy-MM-dd HH:mm:ss"))
                        .withColumn("Ingestion_date", F.to_date("Ingestion_timestamp"))
)

In [0]:
org_table_name = "coreview.bronze.organizations"

if spark.catalog.tableExists(org_table_name):
    org_exploded.write.format("delta").mode("append").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(org_table_name)
else:
    org_exploded.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(org_table_name)


In [0]:
%%sql
select * from coreview.bronze.organizations

In [0]:
signins_path = "/Volumes/coreview/landing/organizations_data/data/signins.csv"

In [0]:
signins_df = spark.read.format('csv').option('header', 'true').load(signins_path)

In [0]:
signins_df = (signins_df.withColumn("Ingestion_timestamp", F.date_format(F.current_timestamp(), "yyyy-MM-dd HH:mm:ss"))
                        .withColumn("Ingestion_date", F.to_date("Ingestion_timestamp"))
)

In [0]:
signins_table_name = "coreview.bronze.signins"

if spark.catalog.tableExists(signins_table_name):
    signins_df.write.format("delta").mode("append").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(signins_table_name)
else:
    signins_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(signins_table_name)


In [0]:
# %%sql
# select * from coreview.bronze.signins